# Orochi API - Custom Rules & Defaults
Interactive Jupyter notebook demonstrating how to manage compiled Custom Rules via the **Orochi Django Ninja REST API** (`/api/customrules/`).

### Topics Covered:
1. Authentication & Session Setup (`/api/auth/`)
2. Listing Custom Rules (`/api/customrules/`)
3. Creating / Merging Custom Rules (`/api/rules/build`)
4. Setting a Custom Rule as Default (`/api/customrules/{id}/default`)
5. Publishing / Sharing Rules with Team Analysts (`/api/customrules/publish`)
6. Downloading Compiled Rules (`/api/customrules/{id}/download`)
7. Deleting Custom Rules (`DELETE /api/customrules/`)

In [ ]:
import getpass
from pprint import pprint

import urllib3
from requests import Session

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

url = "https://localhost"  # Use http://localhost if testing without HTTPS
user = input("Enter Username: ")
password = getpass.getpass("Enter Password: ")

## 1. Authentication

In [ ]:
session = Session()

# 1. Fetch CSRF token from initial GET request
first = session.get(f"{url}/", verify=False)
csrftoken = session.cookies.get("csrftoken") or first.cookies.get("csrftoken", "")

# 2. Login via Django Ninja Auth router
login_payload = {"username": user, "password": password}
headers = {
    "X-CSRFToken": csrftoken,
    "Referer": url,
    "X-Requested-With": "XMLHttpRequest",
}

req = session.post(
    f"{url}/api/auth/",
    json=login_payload,
    cookies=session.cookies,
    headers=headers,
    verify=False,
)

if req.status_code == 200:
    user_data = req.json()
    print(
        f"[+] Successfully authenticated as {user_data.get('username')} (User ID: {user_data.get('id')})"
    )
    # Update session headers so all subsequent requests carry CSRF, Referer, and AJAX headers
    session.headers.update(
        {
            "X-CSRFToken": session.cookies.get("csrftoken", csrftoken),
            "Referer": url,
            "X-Requested-With": "XMLHttpRequest",
        }
    )
else:
    print(f"[-] Login failed ({req.status_code}): {req.text}")

## 2. Listing Existing Custom Rules

In [ ]:
params = {"start": 0, "length": 10, "draw": 1}
res = session.get(f"{url}/api/customrules/", params=params, verify=False)
if res.status_code == 200:
    rules = res.json()
    items = rules.get("data", [])
    total = rules.get("recordsTotal", len(items))
    print(f"Custom Rules Available: {total} (showing {len(items)})")
    if items:
        pprint(items[0])
        rule_pk = items[0]["id"]
        print(f"Existing rule_pk: {rule_pk}")
else:
    print("[-] Error retrieving custom rules:", res.status_code, res.text)

## 3. Create / Build Custom Rule

In [ ]:
# Query available rules to get valid rule IDs for merging
rules_resp = session.get(f"{url}/api/rules/?start=0&length=2&draw=1", verify=False)
if rules_resp.status_code == 200 and rules_resp.json().get("data"):
    rule_ids = [r["id"] for r in rules_resp.json()["data"]]
else:
    rule_ids = [9, 10]

print("Building custom rule from rule IDs:", rule_ids)
build_payload = {
    "rule_ids": rule_ids,
    "rulename": "combined_rule",
}

res = session.post(f"{url}/api/rules/build", json=build_payload, verify=False)
if res.status_code == 200:
    pprint(res.json())
    # Refresh custom rules list to find the newly created rule ID
    cr_refresh = session.get(
        f"{url}/api/customrules/?start=0&length=10&draw=1", verify=False
    ).json()
    for r in cr_refresh.get("data", []):
        if r.get("name") == "combined_rule":
            rule_pk = r["id"]
            break
    else:
        if cr_refresh.get("data"):
            rule_pk = cr_refresh["data"][0]["id"]
        else:
            rule_pk = None
    print(f"[+] Active rule_pk set to: {rule_pk}")
else:
    print("[-] Build error:", res.status_code, res.text)
    # Fallback to existing custom rule
    cr_refresh = session.get(
        f"{url}/api/customrules/?start=0&length=10&draw=1", verify=False
    ).json()
    if cr_refresh.get("data"):
        rule_pk = cr_refresh["data"][0]["id"]
        print(f"[!] Fallback rule_pk: {rule_pk}")

## 4. Set Custom Rule as Default

In [ ]:
res = session.post(f"{url}/api/customrules/{rule_pk}/default", verify=False)
if res.status_code == 200:
    print("[+] Default rule updated:")
    pprint(res.json())
else:
    print("[-] Set default error:", res.status_code, res.text)

## 5. Publish / Share Custom Rule

In [ ]:
pub_payload = {
    "rule_ids": [rule_pk],
    "action": "Publish",
}
res = session.post(f"{url}/api/customrules/publish", json=pub_payload, verify=False)
if res.status_code == 200:
    print("[+] Publish response:")
    pprint(res.json())
else:
    print("[-] Publish error:", res.status_code, res.text)

## 6. Download Custom Rule

In [ ]:
res = session.get(f"{url}/api/customrules/{rule_pk}/download", verify=False)
if res.status_code == 200:
    print(f"[+] Downloaded rule content ({len(res.content)} bytes):")
    print(res.text[:400])
else:
    print("[-] Download error:", res.status_code, res.text)

## 7. Delete Custom Rule

In [ ]:
del_payload = {
    "rule_ids": [rule_pk],
}
res = session.delete(f"{url}/api/customrules/", json=del_payload, verify=False)
if res.status_code == 200:
    print("[+] Delete response:")
    pprint(res.json())
else:
    print("[-] Delete error:", res.status_code, res.text)